# DTU Data Cleaner

In [1]:
import pandas as pd 
import numpy as np
import os 
from pathlib import Path



In [2]:
#CONFIG
DATA_PATH = "/Users/madshaakonsson/Desktop/7-semester/hmm-modelling/data/raw/dtu"
SAVE_PATH = "/Users/madshaakonsson/Desktop/7-semester/hmm-modelling/data/dtu" 

TEST_SIZE = 0.2
TIME_INTERVAL_IN_MINUTES = 30
TAG = "full-series"

In [3]:
def get_data_files(data_path):
    """
    Return all timeseries and feedback CSV files under data_path.

    We clean every room (Room/Corridor/Office/Class/Meeting), so we key off the
    parent folder rather than the file name.
    """
    files = [f for f in Path(data_path).rglob("*.csv") if f.is_file()]
    timeseries_files = sorted(f for f in files if "timeseries" in str(f.parent))
    feedback_files = sorted(f for f in files if "feedback" in str(f.parent))
    return timeseries_files, feedback_files


In [4]:
time_series, feed_back = get_data_files(DATA_PATH)
weather_df = pd.read_csv(os.path.join(DATA_PATH, "weather.csv"))
holiday_23_df = pd.read_excel(os.path.join(DATA_PATH, "holidays_2023.xlsx"), skiprows=1)
holiday_24_df = pd.read_excel(os.path.join(DATA_PATH, "holidays_2024.xlsx"), skiprows=1) 
holiday_df = pd.concat([holiday_23_df, holiday_24_df], ignore_index=True) 
holiday_df.columns = holiday_df.columns.str.strip()  # headers carry stray whitespace

print(f"Found {len(time_series)} time series files and {len(feed_back)} feedback files.")
print(f"Weather data shape: {weather_df.shape}")
print(f"Holiday data shape: {holiday_df.shape}")

print("Sample weather data:")
display(weather_df.head())
print("Sample holiday data:")
display(holiday_df.head())

Found 40 time series files and 16 feedback files.
Weather data shape: (2400, 9)
Holiday data shape: (31, 3)
Sample weather data:


,DateFrom,DateTo,mean_temp,mean_relative_hum,mean_wind_speed,mean_wind_dir,mean_pressure,mean_cloud_cover,mean_radiation
0,2024-02-09 12:00:00+00:00,2024-02-09 13:00:00+00:00,0.1,84.5,5.5,102.0,996.6,94.0,54.2
1,2024-02-09 11:00:00+00:00,2024-02-09 12:00:00+00:00,0.2,79.1,5.3,102.0,997.2,100.0,64.8
2,2024-02-09 10:00:00+00:00,2024-02-09 11:00:00+00:00,0.1,77.9,4.9,99.0,998.0,97.0,56.5
3,2024-02-09 09:00:00+00:00,2024-02-09 10:00:00+00:00,-0.4,81.2,4.2,94.0,998.6,91.0,51.7
4,2024-02-09 08:00:00+00:00,2024-02-09 09:00:00+00:00,-0.6,82.6,4.1,86.0,999.0,91.0,34.6


Sample holiday data:


,DATE,Unnamed: 1,NAME OF HOLIDAY
0,2023-01-01,Sunday,New Year's Day
1,2023-02-20,Monday,Fastelavn
2,2023-04-06,Thursday,Maundy Thursday
3,2023-04-07,Friday,Good Friday
4,2023-04-09,Sunday,Easter Sunday


## Data exploration 

In [5]:
weather_df["DateFrom"] = pd.to_datetime(weather_df["DateFrom"]) 
print("Weather data date range:")
print(weather_df["DateFrom"].min(), weather_df["DateFrom"].max())
print()
print("Missing values in weather data:")
print(weather_df.isnull().sum())

Weather data date range:
2023-11-01 13:00:00+00:00 2024-02-09 12:00:00+00:00

Missing values in weather data:
DateFrom             0
DateTo               0
mean_temp            0
mean_relative_hum    0
mean_wind_speed      0
mean_wind_dir        0
mean_pressure        0
mean_cloud_cover     0
mean_radiation       0
dtype: int64


Weather data is only for every hour, so we will use a linear interpolation to fill in the missing values.

## Cleaning helpers

Aggregate each room's raw (~10 min, irregular) series onto a shared half-hourly
grid, and build a room-independent covariate matrix (interpolated weather +
`is_holiday`/`is_weekend` + cyclic time-of-day `tod_cos`/`tod_sin`) from the
weather and holiday data. Everything is aligned on one tz-naive local-time
`DatetimeIndex`.

In [6]:
# Column groups and resampling settings
SIGNAL_COLS = ["co2", "temperature", "humidity"]
WEATHER_COLS = [
    "mean_temp", "mean_relative_hum", "mean_wind_speed", "mean_wind_dir",
    "mean_pressure", "mean_cloud_cover", "mean_radiation",
]
LOCAL_TZ = "Europe/Copenhagen"
FREQ = f"{TIME_INTERVAL_IN_MINUTES}min"


def build_halfhour_grid(start, end, freq=FREQ):
    """Regular half-hourly, tz-naive DatetimeIndex covering [start, end]."""
    start = pd.Timestamp(start).floor(freq)
    end = pd.Timestamp(end).ceil(freq)
    return pd.date_range(start=start, end=end, freq=freq, name="datetime")

In [7]:
def load_series(path):
    """Load a raw room timeseries CSV, parse datetime and sort chronologically."""
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.sort_values("datetime").set_index("datetime")


def clean_series(df, grid):
    """
    Aggregate a raw room series onto the half-hourly grid.

    - co2 / temperature / humidity -> mean per bin (NaN-safe, so the interleaved
      window-event rows that carry no measurement are simply ignored).
    - Dual-sensor rooms expose <col>_x / <col>_y pairs; average them into <col>.
    - closed (window sensor, when present) -> fraction of the bin reported closed.
    Missing bins are kept as NaN so real gaps stay visible.
    """
    # Collapse dual-sensor pairs (e.g. co2_x / co2_y) into a single mean column.
    for col in SIGNAL_COLS:
        if col not in df.columns:
            pair = [c for c in (f"{col}_x", f"{col}_y") if c in df.columns]
            if pair:
                df = df.assign(**{col: df[pair].mean(axis=1)})

    agg = {c: "mean" for c in SIGNAL_COLS if c in df.columns}

    if "closed" in df.columns:
        closed = df["closed"].map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0})
        df = df.assign(closed_frac=closed)
        agg["closed_frac"] = "mean"

    resampled = df.resample(FREQ).agg(agg)
    return resampled.reindex(grid)

In [8]:
def build_weather_covariates(weather_df, grid):
    """
    Interpolate hourly weather onto the half-hourly grid.

    DateFrom is UTC; convert to local time and drop the tz to match the tz-naive
    room timestamps, then linearly interpolate between the hourly observations.
    """
    w = weather_df.copy()
    w["DateFrom"] = pd.to_datetime(w["DateFrom"], utc=True)
    w["DateFrom"] = w["DateFrom"].dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)
    w = w.sort_values("DateFrom").set_index("DateFrom")[WEATHER_COLS]

    combined = w.reindex(w.index.union(grid))
    combined = combined.interpolate(method="time").ffill().bfill()
    return combined.reindex(grid)


def build_calendar_covariates(holiday_df, grid):
    """
    Calendar covariates on the half-hour grid:
    - is_holiday: 1 on Danish public holidays (joined on calendar date)
    - is_weekend: 1 on Saturday/Sunday
    """
    holidays = set(pd.to_datetime(holiday_df["DATE"]).dt.normalize())
    return pd.DataFrame(
        {
            "is_holiday": grid.normalize().isin(holidays).astype(int),
            "is_weekend": (grid.dayofweek >= 5).astype(int),
        },
        index=grid,
    )


def build_time_of_day_covariates(grid, period=48):
    """
    Cyclic time-of-day covariates: cos/sin of the half-hour-of-day (period 48),
    so 00:00 and 24:00 map to the same point on the circle.
    """
    half_hour = grid.hour * 2 + grid.minute // 30
    angle = 2 * np.pi * half_hour / period
    return pd.DataFrame(
        {"tod_cos": np.cos(angle), "tod_sin": np.sin(angle)},
        index=grid,
    )


def build_covariate_matrix(weather_df, holiday_df, grid):
    """
    Covariate matrix on the half-hour grid: interpolated weather + calendar
    flags (is_holiday, is_weekend) + cyclic time-of-day (tod_cos, tod_sin).
    """
    weather = build_weather_covariates(weather_df, grid)
    calendar = build_calendar_covariates(holiday_df, grid)
    time_of_day = build_time_of_day_covariates(grid)
    return weather.join(calendar).join(time_of_day)

## Build & save cleaned data

Build one shared half-hour grid over the weather span, save the room-independent
covariate matrix once, then aggregate and save each room's half-hourly signal.
All outputs are pandas DataFrames written as CSV under `SAVE_PATH`.

In [9]:
# Shared half-hour grid over the weather span (local, tz-naive) so every room's
# signal and the covariate matrix line up on one index.
weather_local = (
    pd.to_datetime(weather_df["DateFrom"], utc=True)
    .dt.tz_convert(LOCAL_TZ)
    .dt.tz_localize(None)
)
grid = build_halfhour_grid(weather_local.min(), weather_local.max())
print(f"Half-hour grid: {len(grid)} steps, {grid.min()} -> {grid.max()}")

# Covariate matrix (weather + holiday) is room-independent -> build & save once.
covariates = build_covariate_matrix(weather_df, holiday_df, grid)
os.makedirs(SAVE_PATH, exist_ok=True)
covariates.to_csv(os.path.join(SAVE_PATH, "covariates_halfhour.csv"))
print(f"Covariate matrix {covariates.shape} -> covariates_halfhour.csv "
      f"(holidays: {int(covariates['is_holiday'].sum())} bins)")

# Per-room half-hourly signals.
series_dir = os.path.join(SAVE_PATH, "timeseries_halfhour")
os.makedirs(series_dir, exist_ok=True)

for path in time_series:
    df_series = clean_series(load_series(path), grid)
    df_series.to_csv(os.path.join(series_dir, path.name))
    missing = df_series["co2"].isna().mean() if "co2" in df_series.columns else float("nan")
    print(f"{path.stem:20s} {df_series.shape}  co2 missing={missing:5.1%}  -> {path.name}")

Half-hour grid: 4799 steps, 2023-11-01 14:00:00 -> 2024-02-09 13:00:00
Covariate matrix (4799, 11) -> covariates_halfhour.csv (holidays: 192 bins)
Class 026            (4799, 3)  co2 missing= 0.0%  -> Class 026.csv
Corridor 070         (4799, 3)  co2 missing= 0.1%  -> Corridor 070.csv
Corridor 071         (4799, 3)  co2 missing= 0.3%  -> Corridor 071.csv
Corridor 073         (4799, 3)  co2 missing= 0.1%  -> Corridor 073.csv
Meeting room 023     (4799, 3)  co2 missing= 0.1%  -> Meeting room 023.csv
Meeting room 034     (4799, 3)  co2 missing= 0.0%  -> Meeting room 034.csv
Office 017           (4799, 3)  co2 missing= 0.0%  -> Office 017.csv
Office 018           (4799, 4)  co2 missing= 0.1%  -> Office 018.csv
Room 001             (4799, 4)  co2 missing= 0.1%  -> Room 001.csv
Room 003             (4799, 4)  co2 missing= 0.1%  -> Room 003.csv
Room 004             (4799, 4)  co2 missing= 0.1%  -> Room 004.csv
Room 005             (4799, 3)  co2 missing= 0.1%  -> Room 005.csv
Room 007        